In [1]:
# %%

#------------------------------------------------ Begin_Librairie ----------------------------------------

from bs4 import BeautifulSoup

import datetime

import pandas as pd

from pandas import ExcelWriter

from selenium import webdriver

from selenium.webdriver.common.keys import Keys

from selenium.webdriver.common.by import By

import numpy as np

import shutil

from time import sleep

import os

import re

from selenium.webdriver.chrome.service import Service as ChromeService

from webdriver_manager.chrome import ChromeDriverManager

import pdfplumber



In [2]:
# %%

#------------------------------------------------ Begin_ fileName ----------------------------------------

regulatorName = 'JP FSAJP' ## change to current controller name

print(f"Running {regulatorName} Web Scraping Tool v.1.3")

now=datetime.datetime.now()

filename = '{} SQL Ready {}.xlsx'.format(regulatorName, str(now).replace(":",".")[:-7])

scriptfolder = f"C:\\Users\\wuj1\\OneDrive - Moody's\\Desktop\\Regulator\\{regulatorName}"

#scriptfolder=os.path.dirname(os.path.abspath(__file__)) ## to decomment for the production environment

os.chdir(scriptfolder)

writer = ExcelWriter(filename, engine='openpyxl')

tempfolder=os.path.join(scriptfolder, 'tempfolder') #if files are downloaded during the process



if os.path.exists(tempfolder):

    for rem in os.listdir(tempfolder):

        os.remove(os.path.join(tempfolder, rem))

else:

    os.mkdir(tempfolder)




Running JP FSAJP Web Scraping Tool v.1.3


In [3]:
# %%

#------------------------------------------------ Begin_chromedriver ----------------------------------------

#Starting Chrome driver, set to download files in tempfolder

chromeOptions = webdriver.ChromeOptions()

prefs = {"plugins.always_open_pdf_externally": True,

		 "download.prompt_for_download": False,

		 "download.default_directory" : tempfolder,

		 'profile.default_content_setting_values.automatic_downloads': 1}

chromeOptions.add_experimental_option("prefs",prefs)


driver = webdriver.Chrome(options=chromeOptions)

driver.maximize_window()

In [ ]:
regdict={

        'JP FSAJP 1': 'https://www.fsa.go.jp/en/regulated/licensed/city.xls',    

        'JP FSAJP 2': 'https://www.fsa.go.jp/en/regulated/licensed/reg.xlsx', 

        'JP FSAJP 3': 'https://www.fsa.go.jp/en/regulated/licensed/bank_holding.xlsx', 

        'JP FSAJP 4': 'https://www.fsa.go.jp/en/regulated/licensed/s_banks.xls', 

        'JP FSAJP 5': 'https://www.fsa.go.jp/en/regulated/licensed/keito.xlsx', 

        'JP FSAJP 6': 'https://www.fsa.go.jp/en/regulated/licensed/fietb.xlsx', 

        'JP FSAJP 8': 'https://www.fsa.go.jp/en/regulated/licensed/fibo.xlsx', 

        'JP FSAJP 9': 'https://www.fsa.go.jp/en/regulated/licensed/fiisp.xlsx',

        'JP FSAJP 11': 'https://www.fsa.go.jp/en/regulated/licensed/ins_life.xls', 

        'JP FSAJP 12': 'https://www.fsa.go.jp/en/regulated/licensed/ins_nonlife.xlsx', 

        'JP FSAJP 13': 'https://www.fsa.go.jp/en/regulated/licensed/ins_holding.xls', 

        'JP FSAJP 14': 'https://www.fsa.go.jp/en/regulated/licensed/trustcompanies.xlsx', 

        }

Typology ={

        'JP FSAJP 1': 'City Banks and Trust Banks',    

        'JP FSAJP 2': 'Regional Banks & Regional Banks II', 

        'JP FSAJP 3': 'Bank Holding Companies', 

        'JP FSAJP 4': 'Credit Associations (Shinkin Banks)', 

        'JP FSAJP 5': 'Keito Financial Institutions', 

        'JP FSAJP 6': ' Financial Institutions which engage in Trust Business, etc', 

        'JP FSAJP 8': 'Financial Instruments Business Operators', 

        'JP FSAJP 9': 'Financial Instruments Intermediary Service Providers',

        'JP FSAJP 11': 'Life Insurance Companies', 

        'JP FSAJP 12': 'Non-Life Insurance Companies', 

        'JP FSAJP 13': 'Insurance Holding Companies', 

        'JP FSAJP 14': 'Trust Company', 

        }



sqldict = {'bvdid': [], 'priority': [], 'ListLabel': [], 'Typology': [], 'EntryType': [], 'Name': [], 'InternalID_1': [], 'InternalID_1_type': [], 'InternalID_2': [], 

         'InternalID_2_type': [], 'InternalID_3': [], 'InternalID_3_type': [], 'CoType': [], 'License_Type': [], 'Address_1': [], 'Address_2': [], 'City': [], 

         'Zip': [], 'Cntry': [], 'Phone': [], 'Fax': [], 'Website': [], 'Email': [], 'RegulationType': [], 'RegulationTypeCode': [], 'RegulationDate': [], 'CancellationDate': [], 

         'RegCtry': [], 'RegCode' : [], 'ListCode': [], 'ListLanguage': [], 'ListValidityDate': [], 'ListName': [], 'ListProcessDate': [], 'LEI Code': [], 'BIC SWIFT Code': [], 'Name - Mother Company': [],

         'Address_1 - Mother company': [], 'Address_2 -  Mother company': [], 'City - Mother company': [], 'Zip - Mother company': [], 'Cntry - Mother company': [], 

         'Phone - Mother company': [], 'Check': []}

processdate = now.strftime('%Y-%m-%d')


In [5]:
# %%

#------------------------------------------------ Begin_Fouction ----------------------------------------

def bourange_same_length_array(sqldict) :

    maxlen = len(sqldict['ListProcessDate'])

    for key, val in sqldict.items():

        if len(sqldict[key]) != maxlen:

            empty = []

            total_empty = maxlen - len(sqldict[key])

            for i in range(total_empty):

                empty.append('')

            sqldict[key]=sqldict[key]+empty

    return sqldict


In [6]:

#------------------------------------------------ Begin_Main ----------------------------------------

df = pd.DataFrame(sqldict)

for index, reg in enumerate(regdict):
    print(f'[Start New Reg] -- Working with list { reg} --')

    driver.get(regdict[reg])
    sleep(2)
    # soup = BeautifulSoup(driver.page_source, 'html.parser')  
    sleep(5)
    dl_files = [os.path.join(tempfolder, f) for f in os.listdir(tempfolder)]

    if len(dl_files)>0 and not dl_files[0].endswith('.tmp') and not dl_files[0].endswith('.crdownload'):
        sleep(3)
        print('[INFO] -- Check the download file --')

    else:
        print('[INFO] -- Maybe the file link is error, Change to xlsx -- ')
        driver.get(regdict[reg][:]+'x')
        dl_files = [os.path.join(tempfolder, f) for f in os.listdir(tempfolder)]

    # data = pd.read_excel(dl_files[0])
    # data = pd.ExcelFile(dl_files[0])
    # sheet_names = data.sheet_names
    # print(f'[INFO] -- Sheet Numbers: {len(pd.ExcelFile(dl_files[0]).sheet_names)}, Sheet names: {pd.ExcelFile(dl_files[0]).sheet_names}')
    with pd.ExcelFile(dl_files[0]) as xlsx:
        print(f"[INFO] -- Number of sheets: {len(xlsx.sheet_names)}  --")
        print(f"[INFO] -- Number of sheets: {xlsx.sheet_names}  --")
        # Load the first sheet into a DataFrame
        data = pd.read_excel(xlsx, sheet_name=xlsx.sheet_names[0])
    if reg == 'JP FSAJP 1':
        for i in range(len(xlsx.sheet_names)):
            with pd.ExcelFile(dl_files[0]) as xlsx:
                data = pd.read_excel(xlsx, sheet_name=xlsx.sheet_names[i])
            if i == 0:
                df1_1 = pd.DataFrame(sqldict)
                valid_rows = data.iloc[:, :len(data.columns)].notna().all(axis=1)
                if valid_rows.any():
                    header_row_index = valid_rows.idxmax()
                    print(f"First valid header row: {header_row_index}")
                    data.columns = data.iloc[header_row_index].str.replace('\n', ' ')
                    # Remove rows up to the header row
                    data = data.iloc[header_row_index+1:].reset_index(drop=True)
                else:
                    print(f"No valid header row found (first {len(data.columns)} columns contain NaN in every row)")
                data = data[data.columns[1:]]
                print('[INFO] --- Processing data ---')
                df1_1['Name'] = data[data.columns[0]].apply(lambda x: x.replace('\n',' ').strip())
                df1_1['InternalID_1'] = data[data.columns[1]].apply(lambda x: str(x).strip())
                df1_1['Address_1'] = data[data.columns[2]].apply(lambda x: x.replace('\n',' ').strip())
                df1_1['Phone'] = data[data.columns[3]].apply(lambda x: x.replace('\n',' ').strip())
                df1_1['Zip'] = data[data.columns[2]].apply(lambda x: x.split(' ')[-1] if x[0].isdigit() else '')
                df1_1 = df1_1.fillna('')
                df1_1['ListProcessDate'] = processdate
                df1_1['ListName'] = Typology[reg]
                df1_1['RegCtry'] = reg.split(' ')[0]
                df1_1['RegCode'] = reg.split(' ')[1]
                df1_1['ListCode'] = reg.split(' ')[-1]
                df1_1['RegulationType'] = 'Regulated'
                df1_1['InternalID_1_type'] = 'JCN'
                print('[INFO] -- Current File Shape: ',df1_1.shape)
                df = pd.concat([df, df1_1], axis=0)
                print('[INFO] -- Total File Shape: ',df.shape)
            elif i == 1:
                df1_2 = pd.DataFrame(sqldict)
                data.columns = data.iloc[0].str.replace('\n', ' ')
                data = data.iloc[1:, :-1]

                data['Country/Area'] = data['Country/Area'].str.strip()
                data.replace('', np.nan, inplace=True)
                data['Country/Area'] = data['Country/Area'].ffill()

                print('[INFO] --- Processing data ---')
                filtered_data = data[data[data.columns[0]].apply(lambda x: x.replace('\n', ' ').strip() != '')]
                df1_2['Cntry'] = filtered_data[filtered_data.columns[0]].apply(lambda x: x.replace('\n',' ').strip())
                df1_2['Name'] =  filtered_data[filtered_data.columns[1]]
                df1_2['InternalID_1'] =  filtered_data[filtered_data.columns[2]]
                df1_2['Address_1'] =  filtered_data[filtered_data.columns[3]]
                df1_2['Phone'] =  filtered_data[filtered_data.columns[4]]
                df1_2['Zip'] = data[data.columns[3]].apply(lambda x: x.split(' ')[-1] if x[0].isdigit() else '')
                df1_2 = df1_2.fillna('')
                df1_2['ListProcessDate'] = processdate
                df1_2['ListName'] = Typology[reg]
                df1_2['RegCtry'] = reg.split(' ')[0]
                df1_2['RegCode'] = reg.split(' ')[1]
                df1_2['ListCode'] = reg.split(' ')[-1]
                df1_2['RegulationType'] = 'Regulated'
                df1_2['InternalID_1_type'] = 'JCN'
                print('[INFO] -- Current File Shape: ',df1_2.shape)
                df = pd.concat([df, df1_2], axis=0)
                print('[INFO] -- Total File Shape: ',df.shape)

    elif reg == 'JP FSAJP 3' or reg == 'JP FSAJP 6':
        df3 = pd.DataFrame(sqldict)
        valid_rows = data.iloc[:, :len(data.columns)].notna().all(axis=1)
        if valid_rows.any():
            header_row_index = valid_rows.idxmax()  # first True row index
            print(f"First valid header row: {header_row_index}")
            # Set the header to that row
            data.columns = data.iloc[header_row_index].str.replace('\n', ' ')
            # Remove rows up to the header row
            data = data.iloc[header_row_index+1:].reset_index(drop=True)
        else:
            print(f"No valid header row found (first {len(data.columns)} columns contain NaN in every row)")
        
        data = data[data.columns[1:]]
        print('[INFO] --- Processing data ---')

        df3['Name'] = data[data.columns[0]].apply(lambda x: x.replace('\n',' ').strip())
        df3['InternalID_1'] = data[data.columns[1]].apply(lambda x: str(x).strip())
        df3['Address_1'] = data[data.columns[2]].apply(lambda x: x.replace('\n',' ').strip())
        df3['Zip'] = df3['Address_1'].apply(lambda x: x.split(' ')[-1] if len(x.split(' ')[-1])>7 and len(x.split(' ')[-1])<10 else '' )
        df3['Phone'] = data[data.columns[3]].apply(lambda x: x.replace('\n',' ').strip())
        df3 = df3.fillna('')
        df3['ListProcessDate'] = processdate
        df3['ListName'] = Typology[reg]
        df3['RegCtry'] = reg.split(' ')[0]
        df3['RegCode'] = reg.split(' ')[1]
        df3['ListCode'] = reg.split(' ')[-1]
        df3['RegulationType'] = 'Regulated'
        df3['InternalID_1_type'] = 'JCN'
        print('[INFO] -- Current File Shape: ',df3.shape)
        df = pd.concat([df, df3], axis=0)
        print('[INFO] -- Total File Shape: ',df.shape)
    
    elif reg == 'JP FSAJP 2':
        # Check for rows where the first 4 columns have no NaN values
        for i in range(len(xlsx.sheet_names)):
            with pd.ExcelFile(dl_files[0]) as xlsx:
                data = pd.read_excel(xlsx, sheet_name=xlsx.sheet_names[i])
            valid_rows = data.iloc[:, :4].notna().all(axis=1)
            if valid_rows.any():
                header_row_index = valid_rows.idxmax()  # first True row index
                print(f"First valid header row: {header_row_index}")
                # Set the header to that row
                data.columns = data.iloc[header_row_index]
                # Remove rows up to the header row
                data = data.iloc[header_row_index+1:].reset_index(drop=True)
            else:
                print("No valid header row found (first 4 columns contain NaN in every row)")
            df2 = pd.DataFrame(sqldict)    
            print('[INFO] --- Processing data ---')
            data = data[data['JCN'].apply(lambda x: len(str(x)) > 3)]
            df2['Name'] = data[data.columns[1]].apply(lambda x: x.replace('\n',' ').strip())
            df2['InternalID_1'] = data[data.columns[2]].apply(lambda x: str(x).strip())
            df2['Phone'] = data[data.columns[3]].apply(lambda x: x.replace('\n',' ').strip())
            df2 = df2.fillna('')
            df2['ListProcessDate'] = processdate
            df2['ListName'] = Typology[reg]
            df2['RegCtry'] = reg.split(' ')[0]
            df2['RegCode'] = reg.split(' ')[1]
            df2['ListCode'] = reg.split(' ')[-1]
            df2['RegulationType'] = 'Regulated'
            df2['InternalID_1_type'] = 'JCN'
            print('[INFO] -- Current File Shape: ',df2.shape)
            df = pd.concat([df, df2], axis=0)
            print('[INFO] -- Total File Shape: ',df.shape)
            
    elif reg == 'JP FSAJP 4':
        data = data[data.columns[1:]]
        valid_rows = data.iloc[:, :3].notna().all(axis=1)
        header_row_index = valid_rows.idxmax()
        data.columns = data.iloc[header_row_index]
        data = data.iloc[header_row_index+1:].reset_index(drop=True)
        valid_rows = data.iloc[:, :3].notna().all(axis=1)
        data = data[valid_rows]
        data = data[data['JCN'].apply(lambda x: len(str(x)) > 3)]
        df4 = pd.DataFrame(sqldict)
        print('[INFO] --- Processing data ---')

        df4['Name'] = data[data.columns[0]].apply(lambda x: x.replace('\n',' ').strip())
        df4['InternalID_1'] = data[data.columns[1]].apply(lambda x: str(x).strip())
        df4['Phone'] = data[data.columns[2]].apply(lambda x: x.replace('\n',' ').strip())
        df4 = df4.fillna('')
        df4['ListProcessDate'] = processdate
        df4['ListName'] = Typology[reg]
        df4['RegCtry'] = reg.split(' ')[0]
        df4['RegCode'] = reg.split(' ')[1]
        df4['ListCode'] = reg.split(' ')[-1]
        df4['RegulationType'] = 'Regulated'
        df4['InternalID_1_type'] = 'JCN'
        
        print('[INFO] -- Current File Shape: ',df4.shape)
        df = pd.concat([df, df4], axis=0)
        print('[INFO] -- Total File Shape: ',df.shape)
        
    elif reg == 'JP FSAJP 5':
        valid_rows = data.iloc[:, :len(data.columns)].notna().all(axis=1)
        if valid_rows.any():
            header_row_index = valid_rows.idxmax()  # first True row index
            print(f"First valid header row: {header_row_index}")
            # Set the header to that row
            data.columns = data.iloc[header_row_index].str.replace('\n', ' ')
            # Remove rows up to the header row
            data = data.iloc[header_row_index+1:].reset_index(drop=True)
        else:
            print(f"No valid header row found (first {len(data.columns)} columns contain NaN in every row)")
        valid_rows = data.iloc[:, :len(data.columns)].notna().all(axis=1)
        data = data[valid_rows]
        
        df5 = pd.DataFrame(sqldict)
        print('[INFO] --- Processing data ---')
        df5['City'] = data[data.columns[1]].apply(lambda x: x.replace('\n',' ').strip())
        df5['Name'] = data[data.columns[2]].apply(lambda x: x.replace('\n',' ').strip())
        df5['Zip'] =  data[data.columns[3]].apply(lambda x: x.replace('\n',' ').strip())
        df5['Address_1'] = data[data.columns[4]].apply(lambda x: x.replace('\n',' ').strip())
        df5['Phone'] = data[data.columns[5]].apply(lambda x: x.replace('\n',' ').strip())
        df5 = df5.fillna('')
        df5['ListProcessDate'] = processdate
        df5['ListName'] = Typology[reg]
        df5['RegCtry'] = reg.split(' ')[0]
        df5['RegCode'] = reg.split(' ')[1]
        df5['ListCode'] = reg.split(' ')[-1]
        df5['RegulationType'] = 'Regulated'
        print('[INFO] -- Current File Shape: ',df5.shape)
        df = pd.concat([df, df5], axis=0)
        print('[INFO] -- Total File Shape: ',df.shape)
 
    elif reg == 'JP FSAJP 8':
        valid_rows = data.iloc[:, :len(data.columns)].notna().all(axis=1)
        if valid_rows.any():
            header_row_index = valid_rows.idxmax()  # first True row index
            print(f"First valid header row: {header_row_index}")
            # Set the header to that row
            data.columns = data.iloc[header_row_index].str.replace('\n', ' ')
            # Remove rows up to the header row
            data = data.iloc[header_row_index+1:].reset_index(drop=True)
        else:
            print(f"No valid header row found (first {len(data.columns)} columns contain NaN in every row)")
        data = data[data.columns[2:-5]]
        valid_rows = data.iloc[:, :len(data.columns)].notna().all(axis=1)
        data = data[valid_rows]  
        df8 = pd.DataFrame(sqldict)
        print('[INFO] --- Processing data ---')

        df8['Name'] = data[data.columns[0]].apply(lambda x: x.replace('\n',' ').strip())
        df8['InternalID_1'] = data[data.columns[1]].apply(lambda x: str(x).strip())
        df8['Address_1'] = data[data.columns[2]].apply(lambda x: x.replace('\n',' ').strip())
        df8['Phone'] = data[data.columns[3]].apply(lambda x: x.replace('\n',' ').strip())
        #df8['City'] = data[data.columns[2]].apply(lambda x: x.split(',')[-1].strip())
        df8 = df8.fillna('')
        df8['ListProcessDate'] = processdate
        df8['ListName'] = Typology[reg]
        df8['RegCtry'] = reg.split(' ')[0]
        df8['RegCode'] = reg.split(' ')[1]
        df8['ListCode'] = reg.split(' ')[-1]
        df8['RegulationType'] = 'Regulated'
        df8['InternalID_1_type'] = 'JCN'
        print('[INFO] -- Current File Shape: ',df8.shape)
        df = pd.concat([df, df8], axis=0)
        print('[INFO] -- Total File Shape: ',df.shape)    
   
    elif reg == 'JP FSAJP 9':
        valid_rows = data.iloc[:, :len(data.columns)].notna().all(axis=1)
        if valid_rows.any():
            header_row_index = valid_rows.idxmax()  # first True row index
            print(f"First valid header row: {header_row_index}")
            # Set the header to that row
            data.columns = data.iloc[header_row_index].str.replace('\n', ' ')
            # Remove rows up to the header row
            data = data.iloc[header_row_index+1:].reset_index(drop=True)
        else:
            print(f"No valid header row found (first {len(data.columns)} columns contain NaN in every row)")
        data = data[data.columns[2:]]
        valid_rows = data.iloc[:, :len(data.columns)].notna().all(axis=1)
        data = data[valid_rows]  
        df9 = pd.DataFrame(sqldict)
        print('[INFO] --- Processing data ---')

        df9['Name'] = data[data.columns[0]].apply(lambda x: x.replace('\n',' ').strip())
        df9['InternalID_1'] = data[data.columns[1]].apply(lambda x: str(x).strip())
        df9['Address_1'] = data[data.columns[2]].apply(lambda x: x.replace('\n',' ').strip())
        #df9['City'] = data[data.columns[2]].apply(lambda x: x.split(',')[-1].strip())
        df9['Phone'] = data[data.columns[3]].apply(lambda x: x.replace('\n',' ').strip())
        df9['Name - Mother Company'] = data[data.columns[3]].apply(lambda x: x.replace('\n',' ').strip())
        df9 = df9.fillna('')
        df9['ListProcessDate'] = processdate
        df9['ListName'] = Typology[reg]
        df9['RegCtry'] = reg.split(' ')[0]
        df9['RegCode'] = reg.split(' ')[1]
        df9['ListCode'] = reg.split(' ')[-1]
        df9['RegulationType'] = 'Regulated'
        df9['InternalID_1_type'] = 'JCN'
        print('[INFO] -- Current File Shape: ',df9.shape)
        df = pd.concat([df, df9], axis=0)
        print('[INFO] -- Total File Shape: ',df.shape)    

    elif reg == 'JP FSAJP 11':
        data = data[data.columns[2:]]
        valid_rows = data.iloc[:, :len(data.columns)].notna().all(axis=1)
        if valid_rows.any():
            header_row_index = valid_rows.idxmax()  # first True row index
            print(f"First valid header row: {header_row_index}")
            # Set the header to that row
            data.columns = data.iloc[header_row_index].str.replace('\n', ' ')
            # Remove rows up to the header row
            data = data.iloc[header_row_index+1:].reset_index(drop=True)
        else:
            print(f"No valid header row found (first {len(data.columns)} columns contain NaN in every row)")
        valid_rows = data.iloc[:, :len(data.columns)].notna().all(axis=1)
        data = data[valid_rows]
        df11 = pd.DataFrame(sqldict)
        print('[INFO] --- Processing data ---')
        df11['Name'] = data[data.columns[0]].apply(lambda x: x.replace('\n',' ').strip())
        df11['Phone'] = data[data.columns[1]].apply(lambda x: str(x).replace('\n',' ').strip())
        df11 = df11.fillna('')
        df11['ListProcessDate'] = processdate
        df11['ListName'] = Typology[reg]
        df11['RegCtry'] = reg.split(' ')[0]
        df11['RegCode'] = reg.split(' ')[1]
        df11['ListCode'] = reg.split(' ')[-1]
        df11['RegulationType'] = 'Regulated'
        print('[INFO] -- Current File Shape: ',df11.shape)
        df = pd.concat([df, df11], axis=0)

        print('[INFO] -- Total File Shape: ',df.shape)   

    elif reg == 'JP FSAJP 12':
        for i in range(len(xlsx.sheet_names)):
            with pd.ExcelFile(dl_files[0]) as xlsx:
                data = pd.read_excel(xlsx, sheet_name=xlsx.sheet_names[i])
            df12 = pd.DataFrame(sqldict)
            if i == 0:
                data = data[data.columns[1:]]
                valid_rows = data.iloc[:, :len(data.columns)].notna().all(axis=1)
                if valid_rows.any():
                    header_row_index = valid_rows.idxmax()  # first True row index
                    print(f"First valid header row: {header_row_index}")
                    # Set the header to that row
                    data.columns = data.iloc[header_row_index].str.replace('\n', ' ')
                    # Remove rows up to the header row
                    data = data.iloc[header_row_index+1:].reset_index(drop=True)
                else:
                    print(f"No valid header row found (first {len(data.columns)} columns contain NaN in every row)")
                valid_rows = data.iloc[:, :len(data.columns)].notna().all(axis=1)
                data = data[valid_rows]
                print('[INFO] --- Processing data ---')
                df12['Name'] = data[data.columns[0]].apply(lambda x: x.replace('\n',' ').strip())
                df12['Phone'] = data[data.columns[1]].apply(lambda x: str(x).replace('\n',' ').strip())
            elif i == 1:
                valid_rows = data.iloc[:, :len(data.columns)].notna().all(axis=1)
                if valid_rows.any():
                    header_row_index = valid_rows.idxmax()  # first True row index
                    print(f"First valid header row: {header_row_index}")
                    # Set the header to that row
                    data.columns = data.iloc[header_row_index].str.replace('\n', ' ')
                    # Remove rows up to the header row
                    data = data.iloc[header_row_index+1:].reset_index(drop=True)
                else:
                    print(f"No valid header row found (first {len(data.columns)} columns contain NaN in every row)")
                data['Nationality'] = data['Nationality'].str.strip()
                data.replace('', np.nan, inplace=True)
                data['Nationality'] = data['Nationality'].ffill()

                print('[INFO] --- Processing data ---')
                df12['Cntry'] = data[data.columns[0]].apply(lambda x: x.replace('\n',' ').strip())
                df12['Name'] = data[data.columns[1]].apply(lambda x: x.replace('\n',' ').strip())
                df12['Phone'] = data[data.columns[2]].apply(lambda x: str(x).replace('\n',' ').strip())
                
            df12 = df12.fillna('')
            df12['ListProcessDate'] = processdate
            df12['ListName'] = Typology[reg]
            df12['RegCtry'] = reg.split(' ')[0]
            df12['RegCode'] = reg.split(' ')[1]
            df12['ListCode'] = reg.split(' ')[-1]
            df12['RegulationType'] = 'Regulated'
            print('[INFO] -- Current File Shape: ',df12.shape)
            df = pd.concat([df, df12], axis=0)
            print('[INFO] -- Total File Shape: ',df.shape)   

    elif reg == 'JP FSAJP 13':
        data = data[data.columns[1:]]
        valid_rows = data.iloc[:, :len(data.columns)].notna().all(axis=1)
        if valid_rows.any():
            header_row_index = valid_rows.idxmax()  # first True row index
            print(f"First valid header row: {header_row_index}")
            # Set the header to that row
            data.columns = data.iloc[header_row_index].str.replace('\n', ' ')
            # Remove rows up to the header row
            data = data.iloc[header_row_index+1:].reset_index(drop=True)
        else:
            print(f"No valid header row found (first {len(data.columns)} columns contain NaN in every row)")
        valid_rows = data.iloc[:, :len(data.columns)].notna().all(axis=1)
        data = data[valid_rows]

        df13 = pd.DataFrame(sqldict)
        print('[INFO] --- Processing data ---')

        df13['Name'] = data[data.columns[0]].apply(lambda x: x.replace('\n',' ').strip())
        df13['Phone'] = data[data.columns[1]].apply(lambda x: x.replace('\n',' ').strip())
        df13 = df13.fillna('')
        df13['ListProcessDate'] = processdate
        df13['ListName'] = Typology[reg]
        df13['RegCtry'] = reg.split(' ')[0]
        df13['RegCode'] = reg.split(' ')[1]
        df13['ListCode'] = reg.split(' ')[-1]
        df13['RegulationType'] = 'Regulated'
        print('[INFO] -- Current File Shape: ',df13.shape)
        df = pd.concat([df, df13], axis=0)
        print('[INFO] -- Total File Shape: ',df.shape)   

    elif reg == 'JP FSAJP 14':
        data = data[data.columns[1:]]
        valid_rows = data.iloc[:, :len(data.columns)].notna().all(axis=1)
        if valid_rows.any():
            header_row_index = valid_rows.idxmax()  # first True row index
            print(f"First valid header row: {header_row_index}")
            # Set the header to that row
            data.columns = data.iloc[header_row_index].str.replace('\n', ' ')
            # Remove rows up to the header row
            data = data.iloc[header_row_index+1:].reset_index(drop=True)
        else:
            print(f"No valid header row found (first {len(data.columns)} columns contain NaN in every row)")
        valid_rows = data.iloc[:, :len(data.columns)].notna().all(axis=1)
        data = data[valid_rows]

        df14 = pd.DataFrame(sqldict)
        print('[INFO] --- Processing data ---')

        df14['Name'] = data[data.columns[0]].apply(lambda x: x.replace('\n',' ').strip())
        df14['InternalID_1'] = data[data.columns[1]].apply(lambda x: str(x).strip())
        df14['Address_1'] = data[data.columns[2]].apply(lambda x: x.replace('\n',' ').strip())
        df14['Phone'] = data[data.columns[3]].apply(lambda x: x.replace('\n',' ').strip())
        
        df14['Zip'] = data[data.columns[2]].apply(lambda x: x.split(' ')[-1] )
        df14 = df14.fillna('')
        df14['ListProcessDate'] = processdate
        df14['ListName'] = Typology[reg]
        df14['RegCtry'] = reg.split(' ')[0]
        df14['RegCode'] = reg.split(' ')[1]
        df14['ListCode'] = reg.split(' ')[-1]
        df14['RegulationType'] = 'Regulated'
        df14['InternalID_1_type'] = 'JCN'
        print('[INFO] -- Current File Shape: ',df14.shape)

        df = pd.concat([df, df14], axis=0)
        print('[INFO] -- Total File Shape: ',df.shape)   
    sleep(10)
    for rem in os.listdir(tempfolder):
        os.remove(os.path.join(tempfolder, rem))

[Start New Reg] -- Working with list JP FSAJP 11 --
[INFO] -- Check the download file --
[INFO] -- Number of sheets: 1  --
[INFO] -- Number of sheets: ['List']  --
First valid header row: 3
[INFO] --- Processing data ---
[INFO] -- Current File Shape:  (41, 44)
[INFO] -- Total File Shape:  (41, 44)
[Start New Reg] -- Working with list JP FSAJP 12 --
[INFO] -- Maybe the file link is error, Change to xlsx -- 
[INFO] -- Number of sheets: 2  --
[INFO] -- Number of sheets: ['domestic companies', 'Branch Offices']  --
First valid header row: 2
[INFO] --- Processing data ---
[INFO] -- Current File Shape:  (35, 44)
[INFO] -- Total File Shape:  (76, 44)
First valid header row: 2
[INFO] --- Processing data ---
[INFO] -- Current File Shape:  (22, 44)
[INFO] -- Total File Shape:  (98, 44)
[Start New Reg] -- Working with list JP FSAJP 13 --
[INFO] -- Check the download file --
[INFO] -- Number of sheets: 1  --
[INFO] -- Number of sheets: ['Insurance holding companies']  --
First valid header row: 2


In [7]:
# %%

#------------------------------------------------ Begin_writer and save df to excel  ----------------------------------------

os.chdir(scriptfolder)

# df=pd.DataFrame(sqldict)

df.to_excel(writer, 'SQL Ready', index=False)

writer.save()

writer.close()

driver.quit()

sleep(3)



C:\Users\wuj1\AppData\Local\Temp\1\ipykernel_10596\1871660567.py:9: FutureWarning: Starting with pandas version 3.0 all arguments of to_excel except for the argument 'excel_writer' will be keyword-only.
  df.to_excel(writer, 'SQL Ready', index=False)


AttributeError: 'OpenpyxlWriter' object has no attribute 'save'

In [ ]:
df.to_csv('total_zip.csv')

In [ ]:
df3['Address_1'].apply(lambda x: x.split(' ')[-1] if len(x.split(' ')[-1])>7 else '' )

0     100-0005
1     540-8610
2     105-0014
3     104-6228
4     103-0022
5     105-6325
6     100-8233
7     100-0005
8     103-0025
9     107-8472
10    100-8580
11    101-0054
12    100-0004
13    100-8241
14    100-8212
15    102-8660
16    100-0005
17    330-9088
18    320-8610
19    770-8601
20    530-0013
21    790-8514
22    503-0887
23    900-8651
24    892-0828
25    540-8610
26    600-8652
27    107-0062
28    371-8611
29    840-0813
30    520-8686
31    780-8605
32    420-8761
33    310-0021
34    410-8689
35    951-8066
36    260-8720
37    700-8628
38    960-8633
39    630-8677
40    812-0011
41    380-8682
42    860-8615
43    760-8574
44    730-8588
45    810-8727
46    930-8637
47    920-8670
48    330-0854
49    750-8603
50    220-8611
51    900-0034
52    500-8516
53            
54    460-0003
55    103-0028
56    231-8806
57    541-0043
Name: Address_1, dtype: object